# DeepFashion — Quality Check (Step 4)

Validates the curated DeepFashion classroom subset:
- Opens every image with PIL to detect corrupt files
- Prints split / class count table
- Checks class balance (max/min ratio)
- Displays a 4×4 random sample grid per coarse class for visual spot-check

## Parameters

In [ ]:
import pyrootutils

ROOT = pyrootutils.setup_root(
    search_from=".",
    indicator="pyproject.toml",
    project_root_env_var=True,
    dotenv=True,
    pythonpath=True,
    cwd=True,
)

In [ ]:
DATA_DIR = ROOT / "data" / "deepfashion"
SPLITS = ["train", "val", "test"]
GRID_COLS = 4
GRID_ROWS = 4  # images per class in the visual grid

assert DATA_DIR.exists(), f"Dataset directory not found: {DATA_DIR.resolve()}"
print(f"Dataset: {DATA_DIR.resolve()}")

## Imports

In [ ]:
import random
from collections import defaultdict

import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError

## Section 1 — Inventory

Walk the directory tree and index all images by split and class.

In [ ]:
# image_index[split][label] = [path, ...]
image_index = defaultdict(lambda: defaultdict(list))

for split_dir in sorted(DATA_DIR.iterdir()):
    if not split_dir.is_dir() or split_dir.name not in SPLITS:
        continue
    for label_dir in sorted(split_dir.iterdir()):
        if not label_dir.is_dir():
            continue
        for img_path in sorted(label_dir.iterdir()):
            if img_path.suffix.lower() in (".jpg", ".jpeg", ".png"):
                image_index[split_dir.name][label_dir.name].append(img_path)

all_labels = sorted({label for split in image_index.values() for label in split})
print(f"Classes ({len(all_labels)}): {all_labels}")
print(f"Splits found: {sorted(image_index.keys())}")

## Section 2 — Split / Class Count Table

In [ ]:
col_w = 10
header = (
    f"{'label':<20}" + "".join(f"{s:>{col_w}}" for s in SPLITS) + f"{'total':>{col_w}}"
)
print(header)
print("-" * len(header))

totals = defaultdict(int)
grand_total = 0
for label in all_labels:
    counts = [len(image_index[s].get(label, [])) for s in SPLITS]
    row_total = sum(counts)
    grand_total += row_total
    for s, c in zip(SPLITS, counts, strict=False):
        totals[s] += c
    print(
        f"{label:<20}"
        + "".join(f"{c:>{col_w}}" for c in counts)
        + f"{row_total:>{col_w}}"
    )

print("-" * len(header))
print(
    f"{'TOTAL':<20}"
    + "".join(f"{totals[s]:>{col_w}}" for s in SPLITS)
    + f"{grand_total:>{col_w}}"
)

## Section 3 — Class Balance

Compute max/min image count ratio across classes using the training split.

In [ ]:
train_counts = {label: len(image_index["train"].get(label, [])) for label in all_labels}
max_count = max(train_counts.values())
min_count = min(v for v in train_counts.values() if v > 0)
ratio = max_count / min_count

print("Train class image counts:")
for label, count in sorted(train_counts.items(), key=lambda x: -x[1]):
    bar = "█" * (count * 30 // max_count)
    print(f"  {label:<20} {count:>6}  {bar}")

print(f"\nMax/min ratio (train): {max_count}/{min_count} = {ratio:.1f}x")
if ratio > 3:
    print("WARNING: imbalance > 3x — consider adjusting MAX_ITEMS_PER_CLASS in Step 2.")
else:
    print("OK: reasonably balanced.")

## Section 4 — Corrupt File Scan

Attempts to open every image with PIL. Logs any files that cannot be decoded.

In [ ]:
all_paths = [
    p for split in image_index.values() for paths in split.values() for p in paths
]
print(f"Scanning {len(all_paths)} images...")

corrupt = []
for i, path in enumerate(all_paths):
    try:
        with Image.open(path) as im:
            im.verify()
    except (UnidentifiedImageError, Exception) as e:
        corrupt.append((path, str(e)))
    if (i + 1) % 500 == 0:
        print(f"  {i + 1}/{len(all_paths)}")

print(f"\nCorrupt files: {len(corrupt)}")
if corrupt:
    for path, err in corrupt:
        print(f"  {path.relative_to(DATA_DIR)} — {err}")
else:
    print("All images are readable.")

## Section 5 — Visual Grid per Class

Displays a 4×4 random sample from the training split for each coarse class.
Use this to spot wrong labels or bad crops.

In [ ]:
random.seed(0)

for label in all_labels:
    train_paths = image_index["train"].get(label, [])
    if not train_paths:
        print(f"No training images for class: {label}")
        continue

    sample = random.sample(train_paths, min(GRID_COLS * GRID_ROWS, len(train_paths)))
    n = len(sample)
    n_cols = GRID_COLS
    n_rows = (n + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
    fig.suptitle(
        f"DeepFashion classroom · class: {label} · n_train={len(train_paths)}",
        fontsize=13,
        y=1.01,
    )
    axes = axes.flatten() if n_rows * n_cols > 1 else [axes]

    for ax, path in zip(axes, sample, strict=False):
        with Image.open(path) as im:
            ax.imshow(im)
        ax.set_title(path.name[:20], fontsize=7)
        ax.axis("off")

    for ax in axes[len(sample) :]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## Section 6 — ImageFolder Smoke Test

Verifies that all splits load correctly with `torchvision.datasets.ImageFolder`.

In [ ]:
from torchvision.datasets import ImageFolder

for split in SPLITS:
    split_dir = DATA_DIR / split
    if not split_dir.exists():
        print(f"  {split}: directory not found")
        continue
    ds = ImageFolder(str(split_dir))
    print(f"  {split}: {len(ds)} images, classes={ds.classes}")

## Summary

In [ ]:
issues = []
if corrupt:
    issues.append(f"{len(corrupt)} corrupt file(s)")
if ratio > 3:
    issues.append(f"class imbalance ratio {ratio:.1f}x (>3x)")

if issues:
    print(f"QC issues for [deepfashion]: {'; '.join(issues)}")
else:
    print(
        f"QC passed for [deepfashion]: "
        f"{len(all_paths)} images, {len(all_labels)} classes, "
        f"{ratio:.1f}x imbalance, 0 corrupt."
    )